<a href="https://colab.research.google.com/github/Kalyani-asekar/NNDL/blob/main/practical9nndl_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
vocab_size = 1000
embed_dim = 64
hidden_dim = 32
seq_len = 15
num_samples = 500

X = torch.randint(1, vocab_size, (num_samples, seq_len)).to(device)
y = torch.randint(0, 2, (num_samples, 1)).float().to(device)

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        out, hn = self.rnn(embedded)
        out = self.fc(hn.squeeze(0))
        return self.sigmoid(out)

model = SentimentRNN(vocab_size, embed_dim, hidden_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    preds = (model(X) >= 0.5).float()
    acc = (preds == y).float().mean()
    print(f"\nFinal Accuracy: {acc.item() * 100:.2f}%")